# `apptbl-llm-baseline`: off-the-shelf generative LLMs and decision models on the 200-respondent Vaccines subsample

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "apptbl-llm-baseline"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-llm-baseline.*


## Labels and formatting (edit wording here)

In [2]:
SUMMARY_CSV = PAPER_DIR / "llm-baseline" / "results_summary.csv"
PERSONA = "Respondent"

HEADERS = ["Model", "Reasoning effort", "Refusals", "Opt-out", r"$\mathrm{MSE}_\beta \times 10^3$", "MAE", r"Spearman $\rho$", "Sign acc."]

# One table row each, in this order.
# (family, model, config, readout) select the row of results_summary.csv; config None means the model has one config.
# The last two entries are the text shown in the Model and Reasoning effort columns.
TABLE_ROWS = [
    ("Human benchmark", "Survey subsample (200 resp.)", "Observed choices", "Observed one-hot", "Human subsample (200 resp.)", "Observed"),
    ("Generative LLM (Qwen)", "Qwen3.5-0.8B", "Thinking off", "Discrete text", "Qwen3.5-0.8B", "Off"),
    ("Generative LLM (Qwen)", "Qwen3.5-0.8B", "Thinking on", "Discrete text", "Qwen3.5-0.8B", "On"),
    ("Generative LLM (Qwen)", "Qwen3.5-4B", "Thinking off", "Discrete text", "Qwen3.5-4B", "Off"),
    ("Generative LLM (Qwen)", "Qwen3.5-4B", "Thinking on", "Discrete text", "Qwen3.5-4B", "On"),
    ("Generative LLM (Qwen)", "Qwen3.5-9B", "Thinking off", "Discrete text", "Qwen3.5-9B", "Off"),
    ("Generative LLM (Qwen)", "Qwen3.5-9B", "Thinking on", "Discrete text", "Qwen3.5-9B", "On"),
    ("Generative LLM (Gemini)", "Gemini 3.5 Flash-Lite", "Minimal thinking", "Discrete text", "Gemini 3.5 Flash-Lite", "Min"),
    ("Generative LLM (Gemini)", "Gemini 3.5 Flash-Lite", "High thinking", "Discrete text", "Gemini 3.5 Flash-Lite", "High"),
    ("Generative LLM (Gemini)", "Gemini 4 Argon", "Low thinking", "Discrete text", "Gemini 4 Argon", "Low"),
    ("Generative LLM (Gemini)", "Gemini 4 Argon", "High thinking", "Discrete text", "Gemini 4 Argon", "High"),
    ("Base decision model", "KEV-0.8B", None, "Continuous probs", "KEV-0.8B", "Off-the-shelf"),
    ("Base decision model", "KEV-4B", None, "Continuous probs", "KEV-4B", "Off-the-shelf"),
    ("Base decision model", "KEV-9B", None, "Continuous probs", "KEV-9B", "Off-the-shelf"),
]
COLUMN_RULES = [":---", ":---", "---:", "---:", "---:", "---:", "---:", "---:"]

## Build the table

In [3]:
def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table: header row, rule row (e.g. ':---' left, '---:' right), then one line per row."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(column_rules) + "|")
    for row in rows:
        if len(row) != len(headers):
            raise ValueError(f"Row has {len(row)} cells, expected {len(headers)}: {row}")
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [4]:
summary_df = pd.read_csv(SUMMARY_CSV)

rows = []
for family, model, config, readout, shown_model, shown_effort in TABLE_ROWS:
    mask = (summary_df["family"] == family) & (summary_df["model"] == model)
    mask = mask & (summary_df["persona"] == PERSONA) & (summary_df["readout"] == readout)
    if config is not None:
        mask = mask & (summary_df["config"] == config)
    matches = summary_df[mask]
    if len(matches) != 1:
        raise ValueError(f"Expected 1 row in {SUMMARY_CSV} for {model}/{config}, got {len(matches)}")
    result = matches.iloc[0]
    rows.append([
        shown_model,
        shown_effort,
        f"{int(result['refusals'])}",
        f"{result['opt_out_pct']:.1f}%",
        f"{result['beta_mse_x1e3']:.2f}",
        f"{result['beta_mae']:.3f}",
        f"{result['spearman_rho']:.2f}",
        f"{result['sign_acc_pct']:.1f}%",
    ])

table_md = pipe_table(HEADERS, rows, COLUMN_RULES)
display(pd.DataFrame(rows, columns=HEADERS))

,Model,Reasoning effort,Refusals,Opt-out,$\mathrm{MSE}_\beta \times 10^3$,MAE,Spearman $\rho$,Sign acc.
0,Human subsample (200 resp.),Observed,0,20.3%,0.46,0.018,0.89,100.0%
1,Qwen3.5-0.8B,Off,0,59.0%,2.75,0.038,0.82,90.9%
2,Qwen3.5-0.8B,On,19,28.6%,13.82,0.095,0.87,90.9%
3,Qwen3.5-4B,Off,0,53.6%,1.94,0.029,0.95,90.9%
4,Qwen3.5-4B,On,0,4.8%,9.71,0.085,0.98,100.0%
5,Qwen3.5-9B,Off,0,81.8%,1.49,0.027,0.95,90.9%
6,Qwen3.5-9B,On,1,11.4%,6.27,0.066,0.97,100.0%
7,Gemini 3.5 Flash-Lite,Min,0,21.9%,10.64,0.083,0.94,100.0%
8,Gemini 3.5 Flash-Lite,High,0,4.8%,12.28,0.085,0.90,100.0%
9,Gemini 4 Argon,Low,0,12.2%,16.50,0.099,0.93,100.0%


## Save `apptbl-llm-baseline.md` (included by the manuscript)

In [5]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Model | Reasoning effort | Refusals | Opt-out | $\mathrm{MSE}_\beta \times 10^3$ | MAE | Spearman $\rho$ | Sign acc. |
|:---|:---|---:|---:|---:|---:|---:|---:|
| Human subsample (200 resp.) | Observed | 0 | 20.3% | 0.46 | 0.018 | 0.89 | 100.0% |
| Qwen3.5-0.8B | Off | 0 | 59.0% | 2.75 | 0.038 | 0.82 | 90.9% |
| Qwen3.5-0.8B | On | 19 | 28.6% | 13.82 | 0.095 | 0.87 | 90.9% |
| Qwen3.5-4B | Off | 0 | 53.6% | 1.94 | 0.029 | 0.95 | 90.9% |
| Qwen3.5-4B | On | 0 | 4.8% | 9.71 | 0.085 | 0.98 | 100.0% |
| Qwen3.5-9B | Off | 0 | 81.8% | 1.49 | 0.027 | 0.95 | 90.9% |
| Qwen3.5-9B | On | 1 | 11.4% | 6.27 | 0.066 | 0.97 | 100.0% |
| Gemini 3.5 Flash-Lite | Min | 0 | 21.9% | 10.64 | 0.083 | 0.94 | 100.0% |
| Gemini 3.5 Flash-Lite | High | 0 | 4.8% | 12.28 | 0.085 | 0.90 | 100.0% |
| Gemini 4 Argon | Low | 0 | 12.2% | 16.50 | 0.099 | 0.93 | 100.0% |
| Gemini 4 Argon | High | 0 | 9.1% | 16.57 | 0.099 | 0.95 | 100.0% |
| KEV-0.8B | Off-the-shelf | 0 | 32.1% | 5.85 | 0.065 | 0.50 | 72.7% |
| KEV-4B